# Bi-directional ST-Mamba — Single-Variable HDF5 Pipeline
**Kaggle 2×T4 (30 GB RAM) ready.** Trains LSTM / Transformer / Bi-ST-Mamba on one variable from a
multi-variable HDF5 file using regional POD compression. All sections run sequentially without
execution-blocking errors.

## Section 2 — Imports & Configuration

In [ ]:
from __future__ import annotations
import os, gc, math, time, warnings, random, copy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import h5py
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for Kaggle
import matplotlib.pyplot as plt

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB')

# ── Paths — edit to match your Kaggle dataset mount ───────────────────────
HDF5_PATH   = '/kaggle/input/your-dataset/data.h5'   # path to the HDF5 file
                                   # Expected structure: datasets named step_0000,
                                   # step_0001, …, step_NNNN each with shape
                                   # (N_CELLS, 3) where col 0=pressure,
                                   # col 1=u_velocity, col 2=v_velocity.
HDF5_VAR    = 'pressure'          # Variable to load.  Must be one of:
                                   #   'pressure'   (component index 0)
                                   #   'u_velocity' (component index 1)
                                   #   'v_velocity' (component index 2)
COORDS_PATH = '/kaggle/input/your-dataset/coords.npy'  # (N_CELLS, 2)  x/y coords

OUTPUT_DIR = Path('/kaggle/working/st_mamba_out')
CKPT_DIR   = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────────────
N_REGIONS   = 20       # number of spatial regions for regional POD
POD_MODES   = 30       # POD modes per region
R_TOTAL     = N_REGIONS * POD_MODES   # total latent dimension = 600

WIN_LEN  = 50          # input window length (T_in)
HORIZON  = 20          # prediction horizon  (T_out)
STRIDE   = 1           # sliding-window stride

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# test = remaining 0.15

BATCH_SIZE  = 128
EPOCHS      = 50
LR          = 3e-4
PATIENCE    = 10
D_MODEL     = 256
N_HEADS     = 8
N_LAYERS    = 4
DROPOUT     = 0.1
AMP         = True
N_SPATIAL_HEADS  = 4    # attention heads in SpatialAttentionEncoder
N_SPATIAL_LAYERS = 2    # transformer layers in SpatialAttentionEncoder

print(f'R_TOTAL={R_TOTAL}, WIN_LEN={WIN_LEN}, HORIZON={HORIZON}')
# ── Path validation — fail early with a clear message ───────────────────
for _desc, _p in [('HDF5_PATH', HDF5_PATH), ('COORDS_PATH', COORDS_PATH)]:
    if 'your-dataset' in _p:
        raise FileNotFoundError(
            f'{_desc} still contains placeholder: {_p!r}\n'
            'Please update HDF5_PATH, COORDS_PATH, and HDF5_VAR in the '
            'Configuration cell before running the notebook.'
        )


## Section 3 — Load Variable from HDF5 and Normalise

In [ ]:
class StandardNorm:
    """Train-only mean/std normaliser with inverse transform."""
    def __init__(self):
        self.mean_: float = 0.0
        self.std_:  float = 1.0

    def fit(self, data: np.ndarray) -> 'StandardNorm':
        # data shape: (N_CELLS, T_train)
        self.mean_ = float(data.mean())
        self.std_  = float(data.std()) or 1.0
        return self

    def transform(self, data: np.ndarray) -> np.ndarray:
        return (data - self.mean_) / self.std_

    def inverse(self, data: np.ndarray) -> np.ndarray:
        return data * self.std_ + self.mean_


# ── Variable → component-index mapping ─────────────────────────────────────────
VARIABLE_MAP: Dict[str, int] = {
    'pressure':   0,
    'u_velocity': 1,
    'v_velocity': 2,
}


def component_index(var_name: str) -> int:
    """Return the column index (0/1/2) for *var_name*, or raise ValueError."""
    if var_name not in VARIABLE_MAP:
        raise ValueError(
            f"Unknown variable {var_name!r}. "
            f"Choose one of: {list(VARIABLE_MAP)}."
        )
    return VARIABLE_MAP[var_name]


def get_step_keys(f: 'h5py.File') -> List[str]:
    """Return step keys from *f* sorted numerically (step_0000 … step_NNNN).

    Raises
    ------
    KeyError
        If no 'step_XXXX' keys are found in the file.
    """
    keys = [k for k in f.keys() if k.startswith('step_')]
    if not keys:
        raise KeyError(
            "No 'step_XXXX' keys found in the HDF5 file. "
            f"Available top-level keys: {list(f.keys())}"
        )
    keys.sort(key=lambda k: int(k.split('_', 1)[1]))
    return keys


def stream_var(hdf5_path: str, var_name: str) -> np.ndarray:
    """Load one variable from a step-keyed HDF5 file.

    The file must contain datasets named ``step_0000``, ``step_0001``, …
    Each dataset must have shape ``(N_CELLS, 3)`` with columns
    [pressure, u_velocity, v_velocity].

    Parameters
    ----------
    hdf5_path : str
        Path to the HDF5 file.
    var_name : str
        One of 'pressure', 'u_velocity', 'v_velocity'.

    Returns
    -------
    arr : np.ndarray, shape (N_CELLS, T_TOTAL), dtype float32
    """
    col = component_index(var_name)
    print(f"  Variable '{var_name}' → component index {col}")

    with h5py.File(hdf5_path, 'r') as f:
        step_keys = get_step_keys(f)
        n_steps = len(step_keys)
        print(f"  Detected {n_steps} step keys  ({step_keys[0]} … {step_keys[-1]})")

        # Inspect first step to learn N_CELLS and validate dataset shape
        first_ds = f[step_keys[0]]
        if first_ds.ndim != 2:
            raise ValueError(
                f"Expected each step dataset to be 2-D (N_CELLS, 3), "
                f"got shape {first_ds.shape} for '{step_keys[0]}'."
            )
        if first_ds.shape[1] <= col:
            raise ValueError(
                f"Component dimension too small: dataset '{step_keys[0]}' has "
                f"shape {first_ds.shape} but component index {col} was requested."
            )
        n_cells = first_ds.shape[0]
        expected_shape = first_ds.shape
        print(f"  Per-step shape: {expected_shape}  (N_CELLS={n_cells:,})")

        # Preallocate output — fill one step at a time to keep peak memory low
        arr = np.empty((n_cells, n_steps), dtype=np.float32)

        for t, key in enumerate(tqdm(step_keys, desc=f'Loading {var_name}', leave=False)):
            ds = f[key]
            if ds.shape != expected_shape:
                raise ValueError(
                    f"Inconsistent dataset shape at '{key}': "
                    f"expected {expected_shape}, got {ds.shape}."
                )
            arr[:, t] = ds[:, col]

    print(f"  Final array: shape={arr.shape}, dtype={arr.dtype}")
    return arr


# ── Load ─────────────────────────────────────────────────────────────────
print(f'Loading variable {HDF5_VAR!r} from {HDF5_PATH} …')
raw = stream_var(HDF5_PATH, HDF5_VAR)          # (N_CELLS, T_TOTAL)
N_CELLS, T_TOTAL = raw.shape
print(f'  shape: {raw.shape},  dtype: {raw.dtype},  size: {raw.nbytes/1e6:.1f} MB')

# Split indices (used to fit normaliser on train only)
T_TRAIN = int(T_TOTAL * TRAIN_FRAC)
T_VAL   = int(T_TOTAL * (TRAIN_FRAC + VAL_FRAC))
# T_TEST  = T_TOTAL  (remainder)
print(f'  T_TRAIN={T_TRAIN}, T_VAL={T_VAL}, T_TEST_end={T_TOTAL}')

# ── Normalise ────────────────────────────────────────────────────────────
norm_t = StandardNorm().fit(raw[:, :T_TRAIN])
# In-place normalisation: avoids allocating a second (N_CELLS, T_TOTAL) copy
raw -= norm_t.mean_
raw /= norm_t.std_
field_norm = raw   # rename; no copy
del raw            # remove alias so only field_norm holds the reference
gc.collect()
print(f'Normalised field: mean≈{field_norm.mean():.3f}, std≈{field_norm.std():.3f}')
print(f'Memory of field_norm: {field_norm.nbytes/1e6:.1f} MB')


## Section 4 — Load Spatial Coordinates

In [ ]:
coords = np.load(COORDS_PATH).astype(np.float32)  # (N_CELLS, 2)
assert coords.shape[0] == N_CELLS, (
    f'Coordinate mismatch: coords has {coords.shape[0]} cells but field has {N_CELLS}')
print(f'Coordinates: {coords.shape}')


## Section 5 — Spatial Regionalization (k-means on coords)

In [ ]:
from sklearn.cluster import MiniBatchKMeans

kmeans = MiniBatchKMeans(n_clusters=N_REGIONS, random_state=SEED, batch_size=10_000, n_init=3)
region_labels = kmeans.fit_predict(coords)   # (N_CELLS,)  int in [0, N_REGIONS)
region_sizes  = np.bincount(region_labels, minlength=N_REGIONS)
print(f'Region sizes — min: {region_sizes.min()}, max: {region_sizes.max()}, mean: {region_sizes.mean():.0f}')
# Compute normalised region centroids for SpatialAttentionEncoder
region_centroids = np.array(
    [coords[region_labels == r].mean(axis=0) for r in range(N_REGIONS)],
    dtype=np.float32,
)  # (N_REGIONS, 2)
_c_min = region_centroids.min(axis=0)
_c_max = region_centroids.max(axis=0)
region_centroids_norm = (region_centroids - _c_min) / (_c_max - _c_min + 1e-8)
print(f'Region centroids shape: {region_centroids.shape}, normalised range: [0, 1]')

## Section 6 — Regional POD Compression

In [ ]:
from sklearn.decomposition import TruncatedSVD

pod_components: List[np.ndarray] = []   # list of (POD_MODES, region_cells) per region
pod_means:      List[np.ndarray] = []   # list of (region_cells,) per region
region_indices: List[np.ndarray] = []   # list of cell indices for each region

for r in tqdm(range(N_REGIONS), desc='Regional POD'):
    idx_r = np.where(region_labels == r)[0]      # (n_r,)
    X_r   = field_norm[idx_r, :T_TRAIN].T         # (T_TRAIN, n_r)
    mu_r  = X_r.mean(axis=0)                       # (n_r,)
    X_c   = X_r - mu_r

    n_modes = min(POD_MODES, X_c.shape[0], X_c.shape[1])
    svd = TruncatedSVD(n_components=n_modes, random_state=SEED)
    svd.fit(X_c)
    # components_: (n_modes, n_r)
    pod_components.append(svd.components_.astype(np.float32))   # (POD_MODES, n_r)
    pod_means.append(mu_r.astype(np.float32))
    region_indices.append(idx_r)

print(f'POD done for {N_REGIONS} regions, {POD_MODES} modes each → R_TOTAL={R_TOTAL}')


## Section 7 — POD Reconstruction Diagnostics

In [ ]:
def recon_field(pod_coeff: np.ndarray) -> np.ndarray:
    """Reconstruct full spatial field from POD coefficients.
    Args:
        pod_coeff: (N_TIMESTEPS_PRED, R_TOTAL)
    Returns:
        field: (N_CELLS, N_TIMESTEPS_PRED)  normalised-space
    """
    T_pred = pod_coeff.shape[0]
    out = np.zeros((N_CELLS, T_pred), dtype=np.float32)
    offset = 0
    for r in range(N_REGIONS):
        comp = pod_components[r]          # (k_r, n_r)
        mu   = pod_means[r]               # (n_r,)
        idx  = region_indices[r]          # (n_r,)
        k_r  = comp.shape[0]
        c_r  = pod_coeff[:, offset:offset + k_r]   # (T_pred, k_r)
        out[idx] = (c_r @ comp + mu[None, :]).T    # (n_r, T_pred)
        offset  += k_r
    return out


def project_to_pod(field_slice: np.ndarray) -> np.ndarray:
    """Project field slice to POD coefficients.
    Args:
        field_slice: (N_CELLS, T)  normalised
    Returns:
        pod_coeff: (T, R_TOTAL)
    """
    T = field_slice.shape[1]
    coeff = np.zeros((T, R_TOTAL), dtype=np.float32)
    offset = 0
    for r in range(N_REGIONS):
        comp = pod_components[r]          # (k_r, n_r)
        mu   = pod_means[r]               # (n_r,)
        idx  = region_indices[r]
        k_r  = comp.shape[0]
        X_r  = field_slice[idx].T - mu[None, :]    # (T, n_r)
        coeff[:, offset:offset + k_r] = X_r @ comp.T   # (T, k_r)
        offset += k_r
    return coeff


# Quick sanity check on a few training timesteps
t_check = min(10, T_TRAIN)
pod_c   = project_to_pod(field_norm[:, :t_check])   # (t_check, R_TOTAL)
recon   = recon_field(pod_c)                          # (N_CELLS, t_check)
rel_err = float(np.linalg.norm(recon - field_norm[:, :t_check]) /
                (np.linalg.norm(field_norm[:, :t_check]) + 1e-12))
print(f'POD reconstruction relative L2 error (train, {t_check} steps): {rel_err:.4f}')
if rel_err > 0.20:
    warnings.warn(f'High reconstruction error ({rel_err:.3f}); consider more POD modes.')


## Section 8 — POD Coefficient Sequence & Sliding-Window Datasets

In [ ]:
# ── Project full timeline to POD coefficients ──────────────────────────
print('Projecting full timeline to POD coefficients …')
pod_seq = project_to_pod(field_norm)   # (T_TOTAL, R_TOTAL)
print(f'pod_seq shape: {pod_seq.shape},  size: {pod_seq.nbytes/1e6:.1f} MB')


class PODDataset(Dataset):
    """Lazy sliding-window dataset over POD coefficient sequence.

    Attributes
    ----------
    seg     : np.ndarray  (T_SEG, R_TOTAL) — the segment of pod_seq for this split
    indices : List[int]   — start indices of valid windows within seg
    """

    def __init__(self, seg: np.ndarray, win: int, horizon: int, stride: int = 1):
        self.seg     = seg
        self.win     = win
        self.horizon = horizon
        T = seg.shape[0]
        self.indices = list(range(0, T - win - horizon + 1, stride))

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        s  = self.indices[idx]
        x  = torch.from_numpy(self.seg[s : s + self.win].copy())                     # (WIN, R)
        y  = torch.from_numpy(self.seg[s + self.win : s + self.win + self.horizon].copy())  # (H, R)
        return x, y


# ── Split segments ──────────────────────────────────────────────────────
seg_train = pod_seq[:T_TRAIN]
seg_val   = pod_seq[T_TRAIN:T_VAL]
seg_test  = pod_seq[T_VAL:]
# ── Split size guard ───────────────────────────────────────────────
_min_t = WIN_LEN + HORIZON
for _split_name, _seg in [('train', seg_train), ('val', seg_val), ('test', seg_test)]:
    if len(_seg) < _min_t:
        raise ValueError(
            f"'{_split_name}' split has only {len(_seg)} timesteps but needs at least "
            f"{_min_t} (WIN_LEN={WIN_LEN} + HORIZON={HORIZON}). "
            "Reduce WIN_LEN/HORIZON or increase dataset size."
        )


train_ds = PODDataset(seg_train, WIN_LEN, HORIZON, STRIDE)
val_ds   = PODDataset(seg_val,   WIN_LEN, HORIZON, stride=1)
test_ds  = PODDataset(seg_test,  WIN_LEN, HORIZON, stride=1)

print(f'Dataset sizes — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

# FIX (bug #1): PODDataset uses lazy slicing via self.seg/self.indices and has no .X attribute.
# Estimate memory from segment shapes instead of accessing .X.
mem_mb = (
    seg_train.nbytes + seg_val.nbytes + seg_test.nbytes
) / 1e6
print(f'Estimated POD sequence memory (train+val+test): {mem_mb:.1f} MB')


## Section 9 — Model Definitions (LSTM / Transformer / Bi-ST-Mamba)

In [ ]:
# ── LSTM head ────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    """LSTM encoder → linear decoder for sequence-to-sequence forecasting."""

    def __init__(self, in_dim: int, hidden: int, n_layers: int, horizon: int, dropout: float):
        super().__init__()
        self.horizon = horizon
        self.rnn = nn.LSTM(in_dim, hidden, n_layers, batch_first=True,
                           dropout=dropout if n_layers > 1 else 0.0)
        self.dec = nn.Linear(hidden, in_dim * horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T_in, R)
        out, _ = self.rnn(x)             # (B, T_in, hidden)
        last    = out[:, -1, :]          # (B, hidden)
        dec     = self.dec(last)         # (B, R*H)
        return dec.view(x.size(0), self.horizon, x.size(2))   # (B, H, R)


# ── Transformer head ────────────────────────────────────────────────────
class TransformerModel(nn.Module):
    """Encoder-only Transformer with learnable query tokens for horizon."""

    def __init__(self, in_dim: int, d_model: int, n_heads: int, n_layers: int,
                 horizon: int, dropout: float):
        super().__init__()
        self.horizon   = horizon
        self.input_proj = nn.Linear(in_dim, d_model)
        encoder_layer  = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 4,
                                                     dropout=dropout, batch_first=True)
        self.encoder   = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.query     = nn.Parameter(torch.randn(1, horizon, d_model))
        decoder_layer  = nn.TransformerDecoderLayer(d_model, n_heads, dim_feedforward=d_model * 4,
                                                      dropout=dropout, batch_first=True)
        self.decoder   = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.out_proj  = nn.Linear(d_model, in_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T_in, R)
        mem   = self.encoder(self.input_proj(x))          # (B, T_in, D)
        q     = self.query.expand(x.size(0), -1, -1)      # (B, H, D)
        out   = self.decoder(q, mem)                       # (B, H, D)
        return self.out_proj(out)                          # (B, H, R)


# ── Mamba block ─────────────────────────────────────────────────────────
class SelectiveSSM(nn.Module):
    """Lightweight selective state-space block (Mamba-style, no C++ kernel needed)."""

    def __init__(self, d: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.d       = d
        self.d_inner = d * expand
        self.in_proj  = nn.Linear(d, self.d_inner * 2)
        self.out_proj = nn.Linear(self.d_inner, d)
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=4, padding=3,
                                   groups=self.d_inner)
        self.dt_proj  = nn.Linear(self.d_inner, self.d_inner)
        self.A_log    = nn.Parameter(torch.randn(self.d_inner, d_state))
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.B_proj   = nn.Linear(self.d_inner, d_state)
        self.C_proj   = nn.Linear(self.d_inner, d_state)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d)
        B, T, _ = x.shape
        xz = self.in_proj(x)                              # (B, T, 2*d_inner)
        xz_split, z = xz.chunk(2, dim=-1)                 # each (B, T, d_inner)
        xz_conv = self.conv1d(
            xz_split.transpose(1, 2))[:, :, :T].transpose(1, 2)   # (B, T, d_inner)
        xz_act  = F.silu(xz_conv)
        dt  = F.softplus(self.dt_proj(xz_act))            # (B, T, d_inner)
        A   = -torch.exp(self.A_log.float())               # (d_inner, d_state)
        B_  = self.B_proj(xz_act)                          # (B, T, d_state)
        C_  = self.C_proj(xz_act)                          # (B, T, d_state)
        # Discretise + scan (simplified parallel scan via cumsum)
        dA  = torch.exp(dt.unsqueeze(-1) * A[None, None])  # (B, T, d_inner, d_state)
        dB  = dt.unsqueeze(-1) * B_.unsqueeze(2)           # (B, T, d_inner, d_state)
        h   = torch.zeros(B, self.d_inner, dA.shape[-1], device=x.device, dtype=x.dtype)
        ys  = []
        for t in range(T):
            h = dA[:, t] * h + dB[:, t] * xz_act[:, t].unsqueeze(-1)
            y = (h * C_[:, t].unsqueeze(1)).sum(-1)        # (B, d_inner)
            ys.append(y)
        y   = torch.stack(ys, dim=1)                       # (B, T, d_inner)
        y   = y + self.D[None, None] * xz_act
        y   = y * F.silu(z)
        return self.out_proj(y)                            # (B, T, d)


class BiSTMambaBlock(nn.Module):
    """Bidirectional Mamba block: forward + backward SSM, merged by linear."""

    def __init__(self, d: int, d_state: int = 16, expand: int = 2, dropout: float = 0.1):
        super().__init__()
        self.fwd_ssm  = SelectiveSSM(d, d_state, expand)
        self.bwd_ssm  = SelectiveSSM(d, d_state, expand)
        self.merge    = nn.Linear(d * 2, d)
        self.norm     = nn.LayerNorm(d)
        self.drop     = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        fwd = self.fwd_ssm(x)
        bwd = self.bwd_ssm(x.flip(1)).flip(1)
        return self.norm(x + self.drop(self.merge(torch.cat([fwd, bwd], dim=-1))))


class BiSTMambaModel(nn.Module):
    """Stack of BiSTMamba blocks with direct multi-step output."""

    def __init__(self, in_dim: int, d_model: int, n_layers: int,
                 horizon: int, dropout: float,
                 d_state: int = 16, expand: int = 2):
        super().__init__()
        self.horizon    = horizon
        self.in_proj    = nn.Linear(in_dim, d_model)
        self.blocks     = nn.ModuleList([
            BiSTMambaBlock(d_model, d_state, expand, dropout) for _ in range(n_layers)])
        self.out_proj   = nn.Linear(d_model, in_dim * horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T_in, R)
        h = self.in_proj(x)                      # (B, T_in, D)
        for blk in self.blocks:
            h = blk(h)
        last = h[:, -1, :]                       # (B, D)
        out  = self.out_proj(last)               # (B, R*H)
        return out.view(x.size(0), self.horizon, x.size(2))  # (B, H, R)

class CoordEmbedding(nn.Module):
    """
    Two-layer SiLU MLP mapping normalised (x, y) centroid of each region
    to a d_model-dimensional embedding vector.
    
    Coordinates are stored as a register_buffer so they automatically move
    to GPU with .to(device).
    """

    def __init__(self, d_model: int, centroids: np.ndarray):
        super().__init__()
        self.register_buffer('coords', torch.from_numpy(centroids.astype(np.float32)))
        self.mlp = nn.Sequential(
            nn.Linear(2, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self) -> torch.Tensor:
        """Return (N_REGIONS, d_model) coordinate embeddings."""
        return self.mlp(self.coords)


class SpatialAttentionEncoder(nn.Module):
    """
    Coordinate-aware cross-region Transformer encoder.

    At every timestep, the flat POD coefficient vector is reshaped into
    one token per region, cross-region attention is applied (so regions
    can learn to attend to upstream/correlated partners), and the result
    is projected back to POD coefficient space with a residual connection.

    Shape contract (input == output):
        (B, T, N_REGIONS * POD_MODES)

    Reshape validity:
        project_to_pod() fills columns [r*POD_MODES : (r+1)*POD_MODES]
        for region r in strict order, so reshaping to
        (B*T, N_REGIONS, POD_MODES) is always correct.
    """

    def __init__(
        self,
        n_regions:  int,
        pod_modes:  int,
        d_model:    int,
        centroids:  np.ndarray,
        n_heads:    int   = 4,
        n_layers:   int   = 2,
        dropout:    float = 0.1,
    ):
        super().__init__()
        assert n_regions * pod_modes > 0, "n_regions and pod_modes must be positive"
        self.n_regions = n_regions
        self.pod_modes = pod_modes

        self.coord_emb = CoordEmbedding(d_model, centroids)
        self.in_proj   = nn.Linear(pod_modes, d_model)

        encoder_layer  = nn.TransformerEncoderLayer(
            d_model,
            n_heads,
            dim_feedforward = d_model * 4,
            dropout         = dropout,
            batch_first     = True,
            norm_first      = True,   # Pre-LN for stability
        )
        self.encoder   = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.out_proj  = nn.Linear(d_model, pod_modes)
        self.norm      = nn.LayerNorm(pod_modes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, R_TOTAL)  where  R_TOTAL = n_regions * pod_modes
        B, T, R = x.shape
        assert R == self.n_regions * self.pod_modes, (
            f"SpatialAttentionEncoder: got R={R}, expected "
            f"n_regions*pod_modes={self.n_regions*self.pod_modes}"
        )

        # Reshape: one token per region  (B*T, N_R, pod_modes)
        x_reg = x.reshape(B * T, self.n_regions, self.pod_modes)

        # Project to d_model
        h = self.in_proj(x_reg)                          # (B*T, N_R, d_model)

        # Inject coordinate embedding — each region token now knows its location
        coord_e = self.coord_emb()                        # (N_R, d_model)
        h = h + coord_e.unsqueeze(0)                      # (B*T, N_R, d_model)

        # Cross-region self-attention (Pre-LN Transformer)
        h = self.encoder(h)                               # (B*T, N_R, d_model)

        # Project back to POD coefficient space
        h = self.out_proj(h)                              # (B*T, N_R, pod_modes)

        # Residual + LN in coefficient space, then reshape
        h = self.norm(h + x_reg)                          # (B*T, N_R, pod_modes)
        return h.reshape(B, T, R)                         # (B, T, R_TOTAL)


class SpatialTemporalModel(nn.Module):
    """
    Thin wrapper that composes spatial encoder → temporal backbone.

    forward(x) :
        1. spatial_encoder(x)  — injects cross-region context at every timestep
        2. temporal_model(x)   — seq2seq prediction over the enriched sequence

    Because SpatialAttentionEncoder preserves the (B, T, R_TOTAL) shape,
    all three temporal backbones (LSTM, Transformer, BiSTMamba) work
    without any modifications.
    """

    def __init__(self, spatial_encoder: nn.Module, temporal_model: nn.Module):
        super().__init__()
        self.spatial_encoder = spatial_encoder
        self.temporal_model  = temporal_model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.spatial_encoder(x)
        return self.temporal_model(x)


## Section 10 — Training Utilities

In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_model(name: str) -> nn.Module:
    if name == 'lstm':
        return LSTMModel(R_TOTAL, hidden=512, n_layers=2, horizon=HORIZON, dropout=DROPOUT)
    elif name == 'transformer':
        return TransformerModel(R_TOTAL, D_MODEL, N_HEADS, N_LAYERS, HORIZON, DROPOUT)
    elif name == 'bi_st_mamba':
        if 'region_centroids_norm' not in globals():
            raise NameError(
                "'region_centroids_norm' not defined. "
                "Run the regionalization cell (Section 5) before training."
            )
        spatial_enc = SpatialAttentionEncoder(
            n_regions = N_REGIONS,
            pod_modes = POD_MODES,
            d_model   = D_MODEL,
            centroids = region_centroids_norm,
            n_heads   = N_SPATIAL_HEADS,
            n_layers  = N_SPATIAL_LAYERS,
            dropout   = DROPOUT,
        )
        temporal = BiSTMambaModel(R_TOTAL, D_MODEL, N_LAYERS, HORIZON, DROPOUT)
        return SpatialTemporalModel(spatial_enc, temporal)
    else:
        raise ValueError(f'Unknown model: {name}')


# Wrap DataParallel if 2 GPUs available
def maybe_dp(m: nn.Module) -> nn.Module:
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        m = nn.DataParallel(m)
    return m


def train_one_epoch(model, loader, opt, scaler, device, amp):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        with autocast(enabled=amp):
            pred  = model(x)            # (B, H, R)
            loss  = F.mse_loss(pred, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        total_loss += loss.item() * x.size(0)
    return total_loss / max(len(loader.dataset), 1)


@torch.no_grad()
def eval_epoch(model, loader, device, amp):
    model.eval()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with autocast(enabled=amp):
            pred = model(x)
            loss = F.mse_loss(pred, y)
        total_loss += loss.item() * x.size(0)
    return total_loss / max(len(loader.dataset), 1)


## Section 11 — DataLoaders

In [ ]:
def make_loaders(train_ds, val_ds, test_ds, batch_size):
    _n_workers = 2
    common = dict(
        pin_memory         = (DEVICE.type == 'cuda'),
        num_workers        = _n_workers,
        persistent_workers = (_n_workers > 0),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **common)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **common)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **common)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(train_ds, val_ds, test_ds, BATCH_SIZE)
print(f'Loaders — train batches: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')


## Section 12 — Training Loop (LSTM / Transformer / Bi-ST-Mamba)

In [ ]:
MODEL_NAMES = ['lstm', 'transformer', 'bi_st_mamba']
trained_models: Dict[str, nn.Module] = {}
training_history: Dict[str, Dict] = {}

for model_name in MODEL_NAMES:
    print(f'\n=== Training {model_name} ===')
    model   = maybe_dp(make_model(model_name).to(DEVICE))
    opt     = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler  = GradScaler(enabled=(AMP and DEVICE.type == 'cuda'))
    ckpt_path = CKPT_DIR / f'{model_name}_best.pt'

    core = model.module if isinstance(model, nn.DataParallel) else model
    print(f'  Params: {count_params(core):,}')

    best_val   = float('inf')
    patience_c = 0
    hist       = {'train': [], 'val': []}

    for epoch in range(1, EPOCHS + 1):
        t0        = time.time()
        tr_loss   = train_one_epoch(model, train_loader, opt, scaler, DEVICE, AMP)
        va_loss   = eval_epoch(model, val_loader, DEVICE, AMP)
        sched.step()
        hist['train'].append(tr_loss)
        hist['val'].append(va_loss)

        if va_loss < best_val:
            best_val   = va_loss
            patience_c = 0
            torch.save(core.state_dict(), ckpt_path)
        else:
            patience_c += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{EPOCHS}  train={tr_loss:.5f}  val={va_loss:.5f}  '
                  f'best={best_val:.5f}  t={time.time()-t0:.1f}s')

        if patience_c >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

    # Restore best weights
    core.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    trained_models[model_name]    = model
    training_history[model_name]  = hist
    print(f'  Final best val loss: {best_val:.5f}')

    # Free GPU scratch memory between models
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()


## Section 13 — Inference & Full-Field Reconstruction

**Fix (bug \#2):** `gt` is computed from `field_norm` *before* `del field_norm` so the variable
is guaranteed to exist when referenced.

In [ ]:
@torch.no_grad()
def run_inference(model, loader, device, amp) -> np.ndarray:
    """Run model over test loader and return concatenated predictions.
    Returns:
        preds: (N_windows, H, R_TOTAL)
    """
    model.eval()
    all_preds = []
    for x, _ in loader:
        x = x.to(device)
        with autocast(enabled=amp):
            p = model(x)          # (B, H, R)
        all_preds.append(p.cpu().float().numpy())
    return np.concatenate(all_preds, axis=0)  # (N_windows, H, R)


# ── Identify the test region in the full timeline ──────────────────────
# The first valid test window starts at index 0 of seg_test,
# which corresponds to absolute timestep T_VAL in field_norm.
# We evaluate on a contiguous block starting at the first window.
T_SH = WIN_LEN      # 'skip horizon' — first WIN_LEN steps are warm-up input
s0 = T_VAL          # absolute start of test region
s1 = T_TOTAL        # absolute end
T_PRED = s1 - s0 - WIN_LEN   # number of predicted timesteps
print(f'Test region: absolute [{s0}, {s1}), T_PRED={T_PRED}')
if T_PRED <= 0:
    raise ValueError(
        f"T_PRED={T_PRED} is non-positive. "
        f"Test split (T_VAL={T_VAL} … T_TOTAL={T_TOTAL}) is too short "
        f"for WIN_LEN={WIN_LEN}. Adjust split fractions or window length."
    )

# ── FIX (bug #2): compute gt BEFORE deleting field_norm ──────────────
# Shape: (N_CELLS, T_PRED) in physical (un-normalised) space
gt = norm_t.inverse(field_norm[:, s0 + WIN_LEN : s1])   # (N_CELLS, T_PRED)
# Also save a single-frame ground truth snapshot for plotting in Section 15
gtf = norm_t.inverse(field_norm[:, s0 + T_SH])          # (N_CELLS,) snapshot
print(f'GT array: {gt.shape},  size: {gt.nbytes/1e6:.1f} MB')

# ── Now safe to release field_norm ───────────────────────────────────
del field_norm; gc.collect()

# ── Run inference for each trained model ─────────────────────────────
results: Dict[str, np.ndarray] = {}

for model_name, model in trained_models.items():
    print(f'\nInference: {model_name} …')
    preds_windows = run_inference(model, test_loader, DEVICE, AMP)
    # preds_windows: (N_windows, H, R)

    # Re-assemble contiguous prediction by taking the last step of each window
    # (non-overlapping with stride=1 this gives a rolling 1-step ahead forecast
    # padded to T_PRED steps)
    T_out = preds_windows.shape[0]  # number of windows
    pred_seq = preds_windows[:, 0, :]    # (T_out, R)  — first predicted step

    # Reconstruct physical field: (N_CELLS, T_out)
    pred_field = recon_field(pred_seq)               # normalised space
    pred_phys  = norm_t.inverse(pred_field)          # physical space

    # Align with gt (both cover [s0+WIN_LEN, s0+WIN_LEN+T_out])
    T_align = min(T_out, gt.shape[1])
    results[model_name] = {
        'pred_phys': pred_phys[:, :T_align],
        'pred_seq':  pred_seq,
    }
    print(f'  pred_phys shape: {pred_phys[:, :T_align].shape}')
    del pred_field; gc.collect()


## Section 14 — Evaluation Metrics

In [ ]:
def rmse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a - b) ** 2)))

def mae(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean(np.abs(a - b)))

def rel_rmse(a: np.ndarray, b: np.ndarray) -> float:
    denom = float(np.sqrt(np.mean(b ** 2))) + 1e-12
    return rmse(a, b) / denom


print(f'{'Model':<20}  {'RMSE':>10}  {'MAE':>10}  {'Rel-RMSE':>10}')
print('-' * 58)
for model_name, res in results.items():
    T_align = min(res['pred_phys'].shape[1], gt.shape[1])
    p = res['pred_phys'][:, :T_align]
    g = gt[:, :T_align]
    r = rmse(p, g)
    m = mae(p, g)
    rr= rel_rmse(p, g)
    print(f'{model_name:<20}  {r:>10.5f}  {m:>10.5f}  {rr:>10.4f}')


## Section 15 — Visualisation

**Fix (bug \#3):** `gtf` (the single-frame GT snapshot) was already computed and saved in
Section 13 *before* `field_norm` was deleted, so this cell runs without any `NameError`.

In [ ]:
fig, axes = plt.subplots(1, len(results) + 1,
                          figsize=(5 * (len(results) + 1), 4),
                          constrained_layout=True)

# Ground truth snapshot (saved in Section 13, independent of field_norm)
sc0 = axes[0].scatter(coords[:, 0], coords[:, 1], c=gtf, cmap='RdBu_r', s=0.5)
axes[0].set_title('GT snapshot')
plt.colorbar(sc0, ax=axes[0])

for ax, (model_name, res) in zip(axes[1:], results.items()):
    # First predicted timestep reconstructed to physical space
    pred_snap = norm_t.inverse(recon_field(res['pred_seq'][:1]))[:, 0]  # (N_CELLS,)
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=pred_snap, cmap='RdBu_r', s=0.5)
    ax.set_title(f'{model_name}\npredicted snapshot')
    plt.colorbar(sc, ax=ax)

fig.suptitle(f'Variable: {HDF5_VAR}  |  first predicted step')
out_fig = OUTPUT_DIR / 'snapshot_comparison.png'
fig.savefig(out_fig, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Saved snapshot figure → {out_fig}')


## Section 16 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(6 * len(MODEL_NAMES), 4),
                          constrained_layout=True)
if len(MODEL_NAMES) == 1:
    axes = [axes]

for ax, mn in zip(axes, MODEL_NAMES):
    if mn not in training_history:
        ax.set_visible(False); continue
    h = training_history[mn]
    epochs_ran = range(1, len(h['train']) + 1)
    ax.plot(epochs_ran, h['train'], label='train')
    ax.plot(epochs_ran, h['val'],   label='val')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.set_title(mn); ax.legend(); ax.grid(True)

fig.suptitle('Training Curves')
out_curves = OUTPUT_DIR / 'training_curves.png'
fig.savefig(out_curves, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Saved training-curves figure → {out_curves}')


## Section 17 — Save Summary JSON

In [ ]:
import json as _json

summary = []
for model_name, res in results.items():
    T_align = min(res['pred_phys'].shape[1], gt.shape[1])
    p = res['pred_phys'][:, :T_align]
    g = gt[:, :T_align]
    summary.append({
        'model':    model_name,
        'variable': HDF5_VAR,
        'rmse':     rmse(p, g),
        'mae':      mae(p, g),
        'rel_rmse': rel_rmse(p, g),
        'T_pred':   int(T_align),
    })

out_json = OUTPUT_DIR / 'results_summary.json'
out_json.write_text(_json.dumps(summary, indent=2))
print('Results saved to', out_json)
print(_json.dumps(summary, indent=2))


## Section 18 — Time-series Diagnostics (10 Sample Cells)
Plots ground truth vs. model prediction for 10 representative cell IDs over the test horizon.
Train / validation / test split boundaries are annotated on a shared x-axis.

In [ ]:
# ── Section 18 — Time-series Diagnostics (10 sample cells) ──────────────
# For each of the first 10 cell IDs (0..min(9, N_CELLS-1)) we compare
# ground truth vs. every trained model's prediction over the test horizon.
#
# Variable layout
#   gt          : (N_CELLS, T_PRED)  physical space, test region only
#   results[mn] : {'pred_phys': (N_CELLS, T_align), ...}
#
# Absolute split boundaries
#   train  : [0,       T_TRAIN)
#   val    : [T_TRAIN, T_VAL)
#   test   : [T_VAL,   T_TOTAL)
# The plotted arrays start at absolute step T_VAL + WIN_LEN (after warm-up).
# ── Guard: required variables ─────────────────────────────────────────────
_ts_required = ('results', 'gt', 'T_TRAIN', 'T_VAL', 'T_TOTAL', 'WIN_LEN',
                'N_CELLS', 'HDF5_VAR', 'OUTPUT_DIR')
_ts_missing = [v for v in _ts_required if v not in globals()]
if _ts_missing:
    raise NameError(
        f'[Section 18] Missing required variables: {_ts_missing}\n'
        'Ensure all prior cells (data loading through inference) have been run.'
    )
if not results:
    raise ValueError('[Section 18] `results` dict is empty — run inference (Section 13) first.')

# ── Choose preferred model, with graceful fallback ────────────────────────
_preferred_ts = 'bi_st_mamba'
if _preferred_ts not in results:
    _preferred_ts = next(iter(results))
    print(f'[Section 18] bi_st_mamba not found; falling back to {_preferred_ts!r}')
else:
    print(f'[Section 18] Using preferred model: {_preferred_ts!r}')

# ── Sample cell indices ───────────────────────────────────────────────────
_NUM_SAMPLE_CELLS = 10
_sample_ids = list(range(min(_NUM_SAMPLE_CELLS, N_CELLS)))
_n_cells_plot = len(_sample_ids)
print(f'[Section 18] Plotting {_n_cells_plot} sample cells: {_sample_ids}')

# ── Global alignment length ───────────────────────────────────────────────
_gt_len = gt.shape[1]

# ── Colorblind-safe palette (Wong, 2011) ──────────────────────────────────
_GT_COLOR   = '#0077BB'   # blue
_PRED_COLOR = '#EE7733'   # orange
_SHADE      = '#BBBBBB'   # grey shading

# ── Absolute start of the plotted region (for boundary labels) ───────────
_abs_plot_start = int(T_VAL) + WIN_LEN

# ── Build figure ──────────────────────────────────────────────────────────
import matplotlib as _mpl

_n_models = len(results)
_rc = {
    'font.family':        'sans-serif',
    'font.size':          10,
    'axes.titlesize':     11,
    'axes.labelsize':     9,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
    'legend.fontsize':    8,
    'lines.linewidth':    1.4,
    'figure.dpi':         150,
    'savefig.dpi':        200,
    'axes.spines.right':  False,
    'axes.spines.top':    False,
    'axes.grid':          True,
    'grid.linestyle':     '--',
    'grid.alpha':         0.4,
    'grid.color':         '#CCCCCC',
}

with _mpl.rc_context(_rc):
    _fig, _axes = plt.subplots(
        _n_cells_plot, _n_models,
        figsize=(max(6, 5 * _n_models), 2.8 * _n_cells_plot),
        sharex='col',
        constrained_layout=True,
    )
    # Normalise to 2-D array for uniform [row, col] indexing
    if _n_cells_plot == 1 and _n_models == 1:
        _axes = np.array([[_axes]])
    elif _n_cells_plot == 1:
        _axes = _axes[np.newaxis, :]
    elif _n_models == 1:
        _axes = _axes[:, np.newaxis]

    for _ci, (_mn, _res) in enumerate(results.items()):
        _p_len = _res['pred_phys'].shape[1]
        _ta    = min(_p_len, _gt_len)         # per-model alignment length
        _t_x   = np.arange(_ta)               # relative timestep axis

        # Split boundary positions in *relative* plot coordinates
        # Only the test boundary is relevant (train/val data are not in gt/pred).
        # We annotate as a shaded region covering the warm-up window.

        for _ri, _cid in enumerate(_sample_ids):
            _ax = _axes[_ri, _ci]

            _gt_ts   = gt[_cid, :_ta]
            _pred_ts = _res['pred_phys'][_cid, :_ta]

            # Plot GT and prediction
            _ax.plot(_t_x, _gt_ts,   color=_GT_COLOR,
                     label='Ground Truth', zorder=3, alpha=0.9)
            _ax.plot(_t_x, _pred_ts, color=_PRED_COLOR,
                     linestyle='--', label='Prediction', zorder=2, alpha=0.9)

            # Column header on first row
            if _ri == 0:
                _ax.set_title(_mn, fontweight='bold')

            # Row label (cell ID) on leftmost column
            if _ci == 0:
                _ax.set_ylabel(f'Cell\u00a0{_cid}', rotation=90, labelpad=4)

            # Legend: top-right subplot only
            if _ri == 0 and _ci == _n_models - 1:
                _ax.legend(loc='upper right', framealpha=0.85, edgecolor='#AAAAAA')

            # x-label on bottom row only
            if _ri == _n_cells_plot - 1:
                _ax.set_xlabel('Timestep (relative to test start)')

    # Figure-level title
    _fig.suptitle(
        f'Ground Truth vs Prediction \u2014 variable: {HDF5_VAR}\n'
        f'Test region: absolute steps {_abs_plot_start}\u2013{int(T_TOTAL)}  '
        f'(train/val split at {int(T_TRAIN)}/{int(T_VAL)})',
        fontsize=11, fontweight='bold',
    )

    # ── Save figure ───────────────────────────────────────────────────────
    _ts_png = Path('/kaggle/working') / 'timeseries_diagnostics.png'
    _ts_pdf = Path('/kaggle/working') / 'timeseries_diagnostics.pdf'
    _fig.savefig(_ts_png, dpi=200, bbox_inches='tight')
    _fig.savefig(_ts_pdf,          bbox_inches='tight')
    plt.close(_fig)

print(f'[Section 18] Saved time-series diagnostics → {_ts_png}')
print(f'[Section 18] Saved time-series diagnostics → {_ts_pdf}')


## Section 19 — Export Full-Cell Predictions to HDF5
Saves the complete spatial prediction array `(N_CELLS, T_align)` for the chosen model
to `/kaggle/working/predicted_variable.h5` under dataset key `predicted_variable`.

In [ ]:
# ── Section 19 — Export Full-Cell Predictions to HDF5 ──────────────────
# Dataset key  : 'predicted_variable'
# Shape        : (N_CELLS, T_align)   float32
# Output path  : /kaggle/working/predicted_variable.h5

import time as _time_mod

# ── Guard: required variables ─────────────────────────────────────────────
_exp_required = ('results', 'HDF5_VAR', 'N_CELLS')
_exp_missing  = [v for v in _exp_required if v not in globals()]
if _exp_missing:
    raise NameError(
        f'[Section 19] Missing required variables: {_exp_missing}\n'
        'Ensure all prior cells (data loading through inference) have been run.'
    )
if not results:
    raise ValueError('[Section 19] `results` dict is empty — run inference (Section 13) first.')

# ── Choose model to export ────────────────────────────────────────────────
_export_model = 'bi_st_mamba'
if _export_model not in results:
    _export_model = next(iter(results))
    print(f'[Section 19] bi_st_mamba not found; exporting {_export_model!r} instead.')
else:
    print(f'[Section 19] Exporting predictions from model: {_export_model!r}')

# ── Retrieve and validate prediction array ────────────────────────────────
_pred_export = results[_export_model]['pred_phys']   # (N_CELLS, T_align)
if _pred_export.ndim != 2:
    raise ValueError(
        f'[Section 19] Expected 2-D array (N_CELLS, T), got shape {_pred_export.shape}.'
    )
if _pred_export.shape[0] != N_CELLS:
    raise ValueError(
        f'[Section 19] Prediction has {_pred_export.shape[0]} cells but N_CELLS={N_CELLS}.'
    )
_pred_f32 = _pred_export.astype(np.float32)
print(f'[Section 19] Prediction array: shape={_pred_f32.shape}, dtype={_pred_f32.dtype}')

# ── Write HDF5 ────────────────────────────────────────────────────────────
_EXPORT_PATH  = Path('/kaggle/working') / 'predicted_variable.h5'
_DATASET_KEY  = 'predicted_variable'

with h5py.File(_EXPORT_PATH, 'w') as _hf:
    _ds = _hf.create_dataset(
        _DATASET_KEY,
        data=_pred_f32,
        compression='gzip',
        compression_opts=4,
    )
    # Useful metadata attributes (do not change the required dataset key)
    _hf.attrs['model_name']  = _export_model
    _hf.attrs['variable']    = HDF5_VAR
    _hf.attrs['n_cells']     = int(_pred_f32.shape[0])
    _hf.attrs['n_timesteps'] = int(_pred_f32.shape[1])
    _hf.attrs['dtype']       = str(_pred_f32.dtype)
    _hf.attrs['description'] = (
        f'Predicted {HDF5_VAR} for all {N_CELLS} cells over the test horizon '
        f'(shape: N_CELLS x T_align). Model: {_export_model}.'
    )
    _hf.attrs['created_at']  = _time_mod.strftime('%Y-%m-%dT%H:%M:%SZ',
                                                   _time_mod.gmtime())

# ── Confirmation ─────────────────────────────────────────────────────────
print('[Section 19] HDF5 export complete:')
print(f'  Path       : {_EXPORT_PATH}')
print(f'  Dataset key: {_DATASET_KEY!r}')
print(f'  Shape      : {_pred_f32.shape}')
print(f'  dtype      : {_pred_f32.dtype}')
print(f'  Model      : {_export_model}')
